In [6]:
import ollama  
import time 

all_models = [m.model for m in ollama.list().models]


print(f"These are the available models: {(all_models)}")

These are the available models: ['qwen2.5:7b', 'nimble:latest', 'deepseek-r1:latest', 'llama3:latest']


## Deploy Ollama Models 

In [19]:
prompt = "Give me the 5 cities worldwide, with the highest population?"



prompt = """Fasse den test in 2 Sätzen zusammen: Gutes Leben
Gute Bildung ist die Grundlage für ein gutes Leben. Sie
entscheidet über Chancen, über Teilhabe und darüber, wie
selbstbestimmt Kinder und Jugendliche ihren Weg gehen
können. Deshalb machen wir uns stark für Förderung, Erzie-
hung und Bildung der Kinder und Jugendlichen. In den ver-
gangenen Jahren haben wir wichtige Schritte hin zu mehr
Chancengerechtigkeit gemacht. Mecklenburg-Vorpommern
gehört heute bundesweit zu den Ländern mit den besten
Voraussetzungen für chancengerechte Bildung. Das ist ein
Erfolg, auf dem wir aufbauen. Gleichzeitig wissen wir: Gute
Bildung ist kein Zustand, sondern eine dauerhafte Aufgabe.
Noch immer erleben Kinder und Jugendliche Brüche in ihren
Bildungsbiografien – beim Übergang von der Kita in die
Schule, von der Schule in Ausbildung oder Studium. Genau
hier setzen wir an. Wir wollen diese Übergänge als verläss­
liche Brücken gestalten, die Sicherheit geben, Orientierung
bieten und individuelle Entwicklung ermöglichen. Unser
Anspruch ist klar: Jedes Kind soll die Unterstützung bekom-
men, die es braucht. Unabhängig von Herkunft, Wohnort
oder den finanziellen Möglichkeiten der Eltern. Wir wollen,
dass alle Kinder und Jugendlichen ihre Potenziale entfalten,
einen anerkannten Schulabschluss erreichen und gut vor-
bereitet ihren weiteren Lebensweg gehen können.
Bildung ist mehr als Wissen – sie ist der Schlüssel zu einem
selbstbestimmten Leben und zu einer gerechteren Gesell-
schaft. Es ist die Zukunft der Kinder und Jugendlichen, die
wir gestalten.
Sozial. Gerecht.
Antifaschistisch"""

used_model = all_models[0]



def ollama_chat(prompt, model):
    stream = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        stream=True,
    )

    output = ""
    for chunk in stream:
        #print(chunk.message.content, end="", flush=True)
        output += chunk.message.content
        print(chunk.message.content, end="", flush=True)
    print("\n")
    return output

output = ollama_chat(prompt, used_model)
print(f"Final output of {used_model} :  {output}")

Ein gutes Leben beginnt mit guter Bildung, die Kinder und Jugendliche auf selbstbestimmten Weg gehend und mit Chancengerechtigkeit versehen soll. Um dies zu gewährleisten, setzen wir an den Übergängen in der Bildungseinrichtungen an und streben an, dass jede Unterstützung erreicht, die das Kind benötigt, unabhängig von seinen Hintergrundbedingungen.

Final output of qwen2.5:7b :  Ein gutes Leben beginnt mit guter Bildung, die Kinder und Jugendliche auf selbstbestimmten Weg gehend und mit Chancengerechtigkeit versehen soll. Um dies zu gewährleisten, setzen wir an den Übergängen in der Bildungseinrichtungen an und streben an, dass jede Unterstützung erreicht, die das Kind benötigt, unabhängig von seinen Hintergrundbedingungen.


## Use Anthropic model (just Haiku, because it is cheap :P )

In [22]:
import anthropic

client = anthropic.Anthropic()

with open("dokument.txt", encoding="utf-8") as f:
    text = f.read()

response = client.messages.create(
    model="claude-haiku-4-5-20251001",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"Fasse das folgende Dokument in 3-5 Stichpunkten zusammen:\n\n{text}"
    }],
)

print(response.content[0].text)
print(response.usage)  # zeigt Input- und Output-Tokens

FileNotFoundError: [Errno 2] No such file or directory: 'dokument.txt'

### DPI Bundestag Agent 

In [8]:
import os, re, httpx
from pydantic_ai import Agent, ModelRetry
from dotenv import load_dotenv
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


load_dotenv()  # vor allem anderen, was die Keys braucht

DIP = "https://search.dip.bundestag.de/api/v1"
KEY = os.environ["DIP_API_KEY"]

model = OpenAIChatModel(
    "qwen2.5:7b",
    provider=OpenAIProvider(base_url="http://localhost:11434/v1", api_key="ollama"),
)
agent = Agent(model)

@agent.tool_plain
def suche_vorgaenge(titel: str, datum_von: str, datum_bis: str) -> str:
    """Sucht Gesetzgebungsvorgänge im DIP des Bundestags nach Stichwort im Titel.

    Args:
        titel: Stichwort, z.B. "Mindestlohn".
        datum_von: Startdatum, Format YYYY-MM-DD.
        datum_bis: Enddatum, Format YYYY-MM-DD.
    """
    for name, wert in (("datum_von", datum_von), ("datum_bis", datum_bis)):
        if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", wert):
            raise ModelRetry(f"{name} muss YYYY-MM-DD sein, z.B. 2025-01-31")

    r = httpx.get(f"{DIP}/vorgang", timeout=15, headers={"Authorization": f"ApiKey {KEY}"},
                  params={"f.titel": titel, "f.datum.start": datum_von, "f.datum.end": datum_bis})
    if r.status_code == 401:
        raise ModelRetry("API-Key ungültig, Suche nicht möglich.")
    r.raise_for_status()

    docs = r.json()["documents"]
    if not docs:
        raise ModelRetry(f"Keine Treffer für '{titel}'. Versuch ein kürzeres Stichwort.")
    return "\n".join(f"{d['datum']} | {d['vorgangstyp']} | {d['titel']}" for d in docs[:10])



result = await agent.run("Was lief 2026 zum Mindestlohn?")
print(result.output)

                 pydantic-ai v2.51.0 • Python 3.13.11
      / \
     /   \       agent: agent • model: openai:qwen2.5:7b • tools: 1 • capabilities: 0
   /___.___\
  /    |    \    observability: off — see every model and tool call live, with cost
/      |      \    set it up free with Logfire and a GitHub login: https://pydantic.dev/ai-setup.md
`---.._|_..---'    or use any OpenTelemetry backend: https://pydantic.dev/docs/ai/logfire/#otel

                 goes away once observability is on — or PYDANTIC_AI_NO_BANNER=1


Im Jahr 2026 gab es mehrere Angelegenheiten im deutschen Bundestag zum Thema Mindestlohn:

- Am 24. September wurde ein Bericht, Gutachten, oder Programm zum Nationalen Aktionsplan zur Förderung von Tarifverhandlungen in Deutschland diskutiert, der nach Artikel 4 Absatz 2 Satz 2 der EU-Mindestlohn-Richtlinie basiert.
- Zudem wurden Schriftliche Fragen zum Thema Arbeitgeberprüfungen zur Einhaltung des Mindestlohns im ersten Halbjahr 2026 gestellt.
- Im Juli wurden Anträge zum Heimische Landwirtschaft stärken - Branchenspezifischer Mindestlohn für Saisonbeschäftigte in der Landwirtschaft eingebracht.
- Es gab mehrere Kleine Anfragen bezüglich des Mindestlohns, wo verschiedene Bundesländer Nachweise zum Vergleich des Mindestlohns und Kontrollen nach 2025 bereitstellten.


### Tagesschau Agent

In [ ]:
@agent.tool_plain
def suche_nachrichten(suchtext: str) -> str:
    """Sucht aktuelle Nachrichten auf tagesschau.de.

    Args:
        suchtext: Suchbegriff, z.B. "Mindestlohn".
    """
    r = httpx.get("https://www.tagesschau.de/api2u/search/", timeout=15,
                  params={"searchText": suchtext, "pageSize": 10})
    r.raise_for_status()
    treffer = r.json().get("searchResults", [])
    if not treffer:
        raise ModelRetry(f"Keine Treffer für '{suchtext}'. Versuch einen anderen Begriff.")
    return "\n".join(f"{t.get('date', '?')[:10]} | {t.get('title', '?')}" for t in treffer)



In [13]:

result = await agent.run("Was lief im 2026 zum Gaza Streifen ?")
print(result.output)

Tatsächlich scheint Ihre Frage in derzeitigem Kontext nicht exakt zu passen, da aktuelle Nachrichten zum Gaza Streifen in das erste Quartal des Jahres 2026 sich momentan nicht referenzieren lassen. Allerdings kann ich Ihnen gerne im gegenwärtigen Zeitraum nach informellen Nachrichten suchen, ob es zu diesem Thema neue Entwicklungen gibt. Wollen Sie, dass ich das versuche?

Andererseits, können Sie möglicherweise damit fragen, was in den ersten Monaten dieses Jahres (2023) zum Gaza Streifen passiert ist, da aktuelle Nachrichten basierend auf denzeitigem Stand verfügbare Informationen sind. Wie wäre es, wenn wir versuchen es so zu formulieren? Sie können das auch selbst gerne anpassen, falls Sie eine spezifischere Frage haben. 

Soll ich versuchen es so zu formulieren?
